Importing and config

In [1]:
from __future__ import annotations

import platform
import sys
import time
from pathlib import Path
from dataclasses import dataclass

import matplotlib
matplotlib.use("Agg")
import hmmlearn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import gammaln, logsumexp
from scipy.stats import multivariate_normal
from hmmlearn.hmm import GaussianHMM

_CWD = Path.cwd().resolve()

_candidates = [
    _CWD,
    _CWD.parent,
    *_CWD.parents,
]

ROOT = next(
    (
        p for p in _candidates
        if (p / "data").is_dir() or (p / "reports").is_dir()
    ),
    _CWD,
)

print(f"ROOT = {ROOT}")

FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

SEEDS = range(5)

try:
    sys.stdout.reconfigure(encoding="utf-8")
except (AttributeError, ValueError, OSError):
    pass

pd.set_option("display.width", 140)

ROOT = D:\Academic\UET\ATCS\shock-detector


cai dat bocpd

In [ ]:
LAM_DEFAULT = 120
R_MAX_DEFAULT = 600
K_CP_DEFAULT = 10


@dataclass(frozen=True)
class NIGPrior:
    """Prior Normal-Inverse-Gamma: sigma^2 ~ IG(alpha0, beta0), mu | sigma^2 ~ N(mu0, sigma^2/kappa0)."""
    mu0: float = 0.0
    kappa0: float = 1.0
    alpha0: float = 2.0
    beta0: float = 1.0

    @classmethod
    def from_train(cls, x_train, kappa0: float = 1.0, alpha0: float = 2.0) -> "NIGPrior":
        """Prior từ dữ liệu TRAIN. Với alpha0 = 2: E[sigma^2] = beta0/(alpha0-1) = var(train)."""
        x = np.asarray(x_train, dtype=float)
        x = x[np.isfinite(x)]
        return cls(mu0=float(x.mean()), kappa0=kappa0, alpha0=alpha0,
                   beta0=float(x.var()) * (alpha0 - 1.0))


def _student_t_logpdf(x: float, mu, kappa, alpha, beta):
    """log predictive Student-t của NIG: df = 2a, loc = mu, scale^2 = b(k+1)/(a k)."""
    df = 2.0 * alpha
    scale2 = beta * (kappa + 1.0) / (alpha * kappa)
    z2 = (x - mu) ** 2 / scale2
    return (gammaln((df + 1) / 2) - gammaln(df / 2)
            - 0.5 * np.log(np.pi * df * scale2)
            - (df + 1) / 2 * np.log1p(z2 / df))


class BOCPD:
    """BOCPD online. Gọi `update(x_t)` lần lượt; mỗi lần trả đặc trưng tại t."""

    def __init__(self, prior: NIGPrior = NIGPrior(), lam: float = LAM_DEFAULT,
                 r_max: int = R_MAX_DEFAULT, k_cp: int = K_CP_DEFAULT):
        self.prior, self.r_max, self.k_cp = prior, int(r_max), int(k_cp)
        self.log_h = np.log(1.0 / lam)
        self.log_1mh = np.log1p(-1.0 / lam)
        # Khởi tạo (bước 1): P(r_0 = 0) = 1, tham số = prior.
        self.log_r = np.array([0.0])
        self.mu = np.array([prior.mu0]); self.kappa = np.array([prior.kappa0])
        self.alpha = np.array([prior.alpha0]); self.beta = np.array([prior.beta0])

    def update(self, x: float) -> dict:
        x = float(x)
        if not np.isfinite(x):
            raise ValueError("x không hữu hạn: xử lý NaN/gap trước khi đưa vào BOCPD")
        # Bước 3: predictive pi_t^(r) cho x_t theo từng run length của t-1.
        log_pi = _student_t_logpdf(x, self.mu, self.kappa, self.alpha, self.beta)
        # Bước 4 (growth) và bước 5 (changepoint).
        log_growth = self.log_r + log_pi + self.log_1mh
        log_cp = logsumexp(self.log_r + log_pi + self.log_h)
        new_log_r = np.concatenate(([log_cp], log_growth))
        # Bước 6-7: evidence và chuẩn hóa -> P(r_t | x_1..t).
        new_log_r -= logsumexp(new_log_r)
        # Bước 8: cập nhật sufficient statistics (eq. 12-13 dạng NIG).
        p = self.prior
        mu_n = (self.kappa * self.mu + x) / (self.kappa + 1.0)
        beta_n = self.beta + self.kappa * (x - self.mu) ** 2 / (2.0 * (self.kappa + 1.0))
        self.mu = np.concatenate(([p.mu0], mu_n))
        self.kappa = np.concatenate(([p.kappa0], self.kappa + 1.0))
        self.alpha = np.concatenate(([p.alpha0], self.alpha + 0.5))
        self.beta = np.concatenate(([p.beta0], beta_n))
        # Cắt run length dài nhất rồi chuẩn hóa lại (mục 2.4).
        if new_log_r.size > self.r_max + 1:
            new_log_r = new_log_r[: self.r_max + 1]
            new_log_r -= logsumexp(new_log_r)
            for name in ("mu", "kappa", "alpha", "beta"):
                setattr(self, name, getattr(self, name)[: self.r_max + 1])
        self.log_r = new_log_r

        prob = np.exp(new_log_r)
        r = np.arange(prob.size)
        return {
            "p_r0": float(prob[0]),                        # chẩn đoán, = H
            "exp_run": float((r * prob).sum()),            # E[r_t | x_1..t]
            "map_run": int(prob.argmax()),
            "cp_recent": float(prob[: self.k_cp + 1].sum()),  # ĐẠI DIỆN: P(r_t <= k_cp)
        }


def run_bocpd(x, prior: NIGPrior = NIGPrior(), lam: float = LAM_DEFAULT,
              r_max: int = R_MAX_DEFAULT, k_cp: int = K_CP_DEFAULT,
              return_matrix: bool = False):
    """Chạy BOCPD tuần tự trên cả chuỗi. Dòng t chỉ phụ thuộc x[:t+1].

    return_matrix=True trả thêm ma trận P(r_t | x_1..t) (T x (r_max+1)) để vẽ.
    """
    model = BOCPD(prior, lam, r_max, k_cp)
    x = np.asarray(x, dtype=float)
    rows, mat = [], (np.zeros((x.size, r_max + 1)) if return_matrix else None)
    for t, xt in enumerate(x):
        rows.append(model.update(xt))
        if return_matrix:
            p = np.exp(model.log_r)
            mat[t, : p.size] = p
    out_df = pd.DataFrame(rows)
    return (out_df, mat) if return_matrix else out_df

Trien khai HMM gauss nhan dien trang thai chuoi thgian, voi 3 trang thai mac dinh

In [3]:
K_DEFAULT = 3


def _as_2d(X) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    return X[:, None] if X.ndim == 1 else X


def fit_hmm(X_train, n_states: int = K_DEFAULT, lengths=None, n_init: int = 5,
            seed: int = 0, n_iter: int = 200, min_covar: float = 1e-3) -> GaussianHMM:
    """Fit GaussianHMM trên TRAIN, n_init lần khởi tạo, giữ log-likelihood cao nhất."""
    X = _as_2d(X_train)
    best, best_ll = None, -np.inf
    for i in range(n_init):
        m = GaussianHMM(n_components=n_states, covariance_type="full", n_iter=n_iter,
                        min_covar=min_covar, random_state=seed + i, tol=1e-4)
        m.fit(X, lengths)
        ll = m.score(X, lengths)
        if ll > best_ll:
            best, best_ll = m, ll
    return best


def forward_filter(model: GaussianHMM, X, lengths=None, return_loglik: bool = False):
    """Filtered probability P(z_t | x_1..t), shape (T, K). Thứ tự state như model.

    lengths: các chuỗi con độc lập (ví dụ tách theo tuần); mỗi chuỗi bắt đầu
    lại từ startprob_. Đây là thuật toán forward thuần, không có backward pass.
    """
    X = _as_2d(X)
    T, K = X.shape[0], model.n_components
    log_pi = np.log(np.clip(model.startprob_, 1e-300, None))
    log_A = np.log(np.clip(model.transmat_, 1e-300, None))
    covars = model.covars_   # hmmlearn trả dạng full (K, d, d)
    log_b = np.column_stack([
        multivariate_normal.logpdf(X, mean=model.means_[k], cov=covars[k], allow_singular=False)
        .reshape(T) for k in range(K)])

    lengths = [T] if lengths is None else list(lengths)
    assert sum(lengths) == T, "sum(lengths) phải bằng số dòng X"
    log_alpha = np.empty((T, K))
    loglik, start = 0.0, 0
    for n in lengths:
        prev = None
        for t in range(start, start + n):
            if prev is None:
                la = log_pi + log_b[t]
            else:
                la = logsumexp(prev[:, None] + log_A, axis=0) + log_b[t]
            c = logsumexp(la)          # log hệ số scaling c_t (Rabiner eq. 91-93)
            loglik += c
            prev = la - c              # chuẩn hóa -> log P(z_t | x_1..t)
            log_alpha[t] = prev
        start += n
    probs = np.exp(log_alpha)
    return (probs, loglik) if return_loglik else probs


def map_states(model: GaussianHMM, vol_col: int = -1) -> tuple[np.ndarray, pd.DataFrame]:
    """Map state -> nhãn theo variance (mean của cột log vol), tăng dần.

    Trả `order`: order[j] = chỉ số state gốc của nhãn j (0 = vol thấp nhất,
    K-1 = vol cao nhất), và bảng mô tả (mean, var, thời gian ở state 1/(1-a_ii)).
    Dùng: probs_mapped = probs[:, order].
    """
    X_means = model.means_
    d = X_means.shape[1]
    if d == 1:   # 1 chiều: sắp theo variance
        key = model.covars_[:, 0, 0]
    else:        # nhiều chiều: sắp theo mean của cột log vol
        key = X_means[:, vol_col]
    order = np.argsort(key)
    K = model.n_components
    names = {2: ["low", "high"], 3: ["low", "mid", "high"]}.get(K, [f"s{j}" for j in range(K)])
    rows = []
    for j, s in enumerate(order):
        row = {"label": names[j], "orig_state": int(s),
               "expected_duration": 1.0 / max(1.0 - model.transmat_[s, s], 1e-12)}
        for c in range(d):
            row[f"mean_{c}"] = X_means[s, c]
            row[f"var_{c}"] = model.covars_[s, c, c]
        rows.append(row)
    return order, pd.DataFrame(rows)


def regime_feature(model: GaussianHMM, X, lengths=None, vol_col: int = -1) -> np.ndarray:
    """ĐẠI DIỆN họ 5: P(state vol cao nhất | x_1..t) -- filtered, đã map nhãn."""
    order, _ = map_states(model, vol_col)
    return forward_filter(model, X, lengths)[:, order[-1]]

Tao du lieu simulation co changepoint, regime an, danh gia kha nang phat hien cua hmm va bocpd

In [4]:
BOCPD_SEGMENTS = [(0.0, 1.0), (2.0, 1.0), (2.0, 0.3), (-1.0, 0.3), (-1.0, 1.5)]
BOCPD_SEG_LEN = 250


def simulate_changepoints(segments=BOCPD_SEGMENTS, seg_len=BOCPD_SEG_LEN, seed=0):
    """Trả (x, cps): cps = chỉ số quan sát ĐẦU TIÊN của mỗi đoạn mới."""
    rng = np.random.default_rng(seed)
    x = np.concatenate([rng.normal(m, s, seg_len) for m, s in segments])
    cps = np.arange(1, len(segments)) * seg_len
    return x, cps

HMM_MEANS = np.array([[0.00, -0.8], [0.02, 0.0], [-0.15, 1.0]])
HMM_SDS = np.array([[0.30, 0.25], [0.60, 0.25], [1.20, 0.30]])
HMM_TRANS = np.array([[0.97, 0.02, 0.01],
                      [0.02, 0.96, 0.02],
                      [0.02, 0.04, 0.94]])


def simulate_hmm(T=3000, seed=0, means=HMM_MEANS, sds=HMM_SDS, trans=HMM_TRANS):
    """Trả (X (T,2), z (T,)) với z là regime thật (0 = low, 2 = high)."""
    rng = np.random.default_rng(seed)
    K = trans.shape[0]
    z = np.empty(T, dtype=int)
    z[0] = rng.integers(K)
    for t in range(1, T):
        z[t] = rng.choice(K, p=trans[z[t - 1]])
    X = means[z] + sds[z] * rng.standard_normal((T, means.shape[1]))
    return X, z


def detect_alarms(map_run, min_len=1):
    """Báo động khi MAP run length TỤT (một đoạn mới trở thành giả thuyết MAP).

    Vị trí change point ước lượng = t - map_run[t] + 1. Bỏ qua tụt về run < min_len
    (chưa đủ bằng chứng). Chỉ dùng để đánh giá trên giả lập; trên dữ liệu thật
    đặc trưng cp_recent đi thẳng vào logistic, không cần ngưỡng.
    """
    map_run = np.asarray(map_run)
    t = np.arange(map_run.size)
    drop = np.r_[False, map_run[1:] < map_run[:-1]] & (map_run >= min_len)
    t_alarm = t[drop]
    return t_alarm, t_alarm - map_run[t_alarm] + 1


def match_alarms(t_alarm, loc, cps, max_delay=50, loc_tol=5):
    """Ghép mỗi cp thật với báo động đầu tiên có t >= cp, t - cp <= max_delay và
    |vị trí ước lượng - cp| <= loc_tol. Trả (delays, loc_errors, n_false)."""
    delays, errs, used = [], [], set()
    for c in cps:
        hit = [i for i in range(len(t_alarm))
               if c <= t_alarm[i] <= c + max_delay and abs(loc[i] - c) <= loc_tol]
        if hit:
            used.update(hit)
            delays.append(int(t_alarm[hit[0]] - c)); errs.append(int(loc[hit[0]] - c))
        else:
            delays.append(None); errs.append(None)
    n_false = len(t_alarm) - len(used)
    return delays, errs, n_false

In [5]:
import io
buf = io.StringIO()


def out(*a):
    s = " ".join(str(x) for x in a)
    print(s)
    buf.write(s + "\n")


def section(title):
    out("\n" + "=" * 70 + f"\n## {title}\n" + "=" * 70)

In [6]:
section("0. MÔI TRƯỜNG")
out(f"python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} "
    f"| hmmlearn {hmmlearn.__version__} | matplotlib {matplotlib.__version__}")


## 0. MÔI TRƯỜNG
python 3.12.10 | numpy 2.5.4 | pandas 3.0.6 | hmmlearn 0.3.3 | matplotlib 3.11.2


In [7]:
section("A. BOCPD (Adams & MacKay 2007) trên chuỗi giả lập")
x0, cps = simulate_changepoints(seed=0)
# Prior từ một chuỗi "train" riêng (mô phỏng tháng 02), KHÔNG lấy từ chuỗi đang chạy.
prior = NIGPrior.from_train(np.random.default_rng(99).normal(0, 1, 500))
out(f"Chuỗi: {len(x0)} bước, cp thật tại {[int(c) for c in cps]}; đoạn (mean, sd): "
    "(0,1) | (2,1) | (2,0.3) | (-1,0.3) | (-1,1.5)")
out(f"Loại cp: 1 đổi mean, 2 giảm var, 3 đổi mean, 4 tăng var. Prior: {prior}")
out(f"Hazard 1/{LAM_DEFAULT}, r_max 600")

t0 = time.perf_counter()
f0, mat0 = run_bocpd(x0, prior, return_matrix=True)
dt = (time.perf_counter() - t0) / len(x0) * 1e3
out(f"\n[A1] P(r_t=0) min/max = {f0.p_r0.min():.6f}/{f0.p_r0.max():.6f}; H = {1/LAM_DEFAULT:.6f}"
    "\n     -> đúng như paper_notes: P(r=0) ≡ H, không mang thông tin, chỉ xuất để chẩn đoán.")


## A. BOCPD (Adams & MacKay 2007) trên chuỗi giả lập
Chuỗi: 1250 bước, cp thật tại [250, 500, 750, 1000]; đoạn (mean, sd): (0,1) | (2,1) | (2,0.3) | (-1,0.3) | (-1,1.5)
Loại cp: 1 đổi mean, 2 giảm var, 3 đổi mean, 4 tăng var. Prior: NIGPrior(mu0=0.021150958644414013, kappa0=1.0, alpha0=2.0, beta0=1.0423010057306108)
Hazard 1/120, r_max 600

[A1] P(r_t=0) min/max = 0.008333/0.008333; H = 0.008333
     -> đúng như paper_notes: P(r=0) ≡ H, không mang thông tin, chỉ xuất để chẩn đoán.


In [8]:
rows = []
for lam in (60, 120, 250):
    for s in SEEDS:
        x, _ = simulate_changepoints(seed=s)
        f = run_bocpd(x, prior, lam=lam)
        ta, loc = detect_alarms(f.map_run)
        d, e, nf = match_alarms(ta, loc, cps)
        rows.append({"lam": lam, "seed": s, **{f"delay_cp{i+1}": d[i] for i in range(4)},
                     "max_|loc_err|": max(abs(v) for v in e if v is not None),
                     "n_detected": sum(v is not None for v in d), "false_alarms": nf})
det = pd.DataFrame(rows)
out("\n[A2] Phát hiện (báo động = MAP run length tụt; vị trí = t - MAP + 1;"
    "\n     khớp nếu trễ ≤ 50 và |sai số vị trí| ≤ 5)")
out(det[det.lam == LAM_DEFAULT].drop(columns="lam").to_string(index=False))
out("\nĐộ nhạy hazard (trung bình 5 seed):")
out(det.groupby("lam")[["delay_cp1", "delay_cp2", "delay_cp3", "delay_cp4",
                        "n_detected", "false_alarms"]].mean().round(2).to_string())


[A2] Phát hiện (báo động = MAP run length tụt; vị trí = t - MAP + 1;
     khớp nếu trễ ≤ 50 và |sai số vị trí| ≤ 5)
 seed  delay_cp1  delay_cp2  delay_cp3  delay_cp4  max_|loc_err|  n_detected  false_alarms
    0          2         27          0          0              1           4             8
    1          2         34          0          1              2           4             1
    2          6         23          0          1              1           4             0
    3          2         26          0          0              1           4             2
    4          2         25          0          2              5           4             6

Độ nhạy hazard (trung bình 5 seed):
     delay_cp1  delay_cp2  delay_cp3  delay_cp4  n_detected  false_alarms
lam                                                                      
60         2.2       25.6        0.0        0.8         4.0          10.4
120        2.8       27.0        0.0        0.8         4.0           3.4
250 

In [9]:
out("\n[A3] Chọn k cho đại diện cp_recent = P(r_t ≤ k) (trung bình 5 seed)")
krows = []
for k in (5, 10, 20):
    peaks, base = [], []
    for s in SEEDS:
        x, _ = simulate_changepoints(seed=s)
        c = run_bocpd(x, prior, k_cp=k).cp_recent.to_numpy()
        peaks.append([c[cp:cp + 60].max() for cp in cps])
        far = np.ones(len(x), bool)
        for cp in cps:
            far[max(cp - 5, 0):cp + 60] = False
        far[:60] = False
        base.append([np.median(c[far]), np.quantile(c[far], 0.99)])
    p, b = np.mean(peaks, 0), np.mean(base, 0)
    krows.append({"k": k, **{f"peak_cp{i+1}": round(p[i], 3) for i in range(4)},
                  "median_xa_cp": round(b[0], 3), "q99_xa_cp": round(b[1], 3)})
out(pd.DataFrame(krows).to_string(index=False))
out("Ghi chú: k nhỏ chỉ 'thấy' cp được phát hiện trong ≤ k bước; k lớn tăng mức nền.")
out(f"\n[A4] Thời gian: {dt:.3f} ms/bước (G5 ~42.000 khối M5 -> ~{dt*42:.0f} s/lượt)")


[A3] Chọn k cho đại diện cp_recent = P(r_t ≤ k) (trung bình 5 seed)
 k  peak_cp1  peak_cp2  peak_cp3  peak_cp4  median_xa_cp  q99_xa_cp
 5     0.597     0.017       1.0     0.880         0.015      0.174
10     0.941     0.020       1.0     0.993         0.018      0.272
20     0.997     0.069       1.0     1.000         0.021      0.428
Ghi chú: k nhỏ chỉ 'thấy' cp được phát hiện trong ≤ k bước; k lớn tăng mức nền.

[A4] Thời gian: 0.394 ms/bước (G5 ~42.000 khối M5 -> ~17 s/lượt)


In [10]:
fig, ax = plt.subplots(3, 1, figsize=(11, 8), sharex=True,
                       gridspec_kw={"height_ratios": [1.2, 1.5, 1]})
ax[0].plot(x0, lw=0.6, color="k")
ta, loc = detect_alarms(f0.map_run)
for c in cps:
    for a in ax:
        a.axvline(c, color="tab:red", ls="--", lw=0.8)
ax[0].set_ylabel("x_t (log RV chuẩn hóa, giả lập)")
ax[0].set_title("BOCPD trên dữ liệu giả lập — đường đỏ: change point thật")
T = len(x0)
ax[1].imshow(np.log10(mat0.T[:400] + 1e-12), aspect="auto", origin="lower",
             cmap="gray_r", vmin=-6, vmax=0, extent=[0, T, 0, 400])
ax[1].plot(f0.map_run, color="tab:blue", lw=0.8, label="MAP run length")
ax[1].set_ylabel("run length r_t"); ax[1].legend(loc="upper left")
ax[1].set_title("log10 P(r_t | x_1..t) (đậm = xác suất cao)")
for k, col in ((5, "tab:orange"), (10, "tab:green"), (20, "tab:purple")):
    ax[2].plot(run_bocpd(x0, prior, k_cp=k).cp_recent, lw=0.8, color=col, label=f"P(r_t ≤ {k})")
ax[2].set_ylabel("cp_recent"); ax[2].set_xlabel("t (bước)"); ax[2].legend(loc="upper left", ncol=3)
fig.tight_layout(); fig.savefig(FIG / "g2_bocpd_sim.png", dpi=150); plt.close(fig)

In [11]:
section("B. Gaussian HMM 3 state + forward filter tự viết")
X, z = simulate_hmm(T=3000, seed=0)
X_tr, X_te, z_te = X[:2000], X[2000:], z[2000:]
model = fit_hmm(X_tr, n_states=3, n_init=5, seed=0)   # fit CHỈ trên train
order, table = map_states(model)
out("Quan sát 2 chiều (return, log vol). Fit trên 2.000 bước đầu, filter trên 1.000 bước sau.")
out("\n[B1] Tham số thật (theo thứ tự low/mid/high):")
out(pd.DataFrame({"mean_ret": HMM_MEANS[:, 0], "mean_logvol": HMM_MEANS[:, 1],
                  "var_ret": HMM_SDS[:, 0] ** 2, "var_logvol": HMM_SDS[:, 1] ** 2},
                 index=["low", "mid", "high"]).round(3).to_string())
out("\nƯớc lượng sau khi map state theo mean log vol (cột 1):")
out(table.round(3).to_string(index=False))


## B. Gaussian HMM 3 state + forward filter tự viết
Quan sát 2 chiều (return, log vol). Fit trên 2.000 bước đầu, filter trên 1.000 bước sau.

[B1] Tham số thật (theo thứ tự low/mid/high):
      mean_ret  mean_logvol  var_ret  var_logvol
low       0.00         -0.8     0.09       0.062
mid       0.02          0.0     0.36       0.062
high     -0.15          1.0     1.44       0.090

Ước lượng sau khi map state theo mean log vol (cột 1):
label  orig_state  expected_duration  mean_0  var_0  mean_1  var_1
  low           0             30.954   0.024  0.088  -0.803  0.059
  mid           1             18.766   0.026  0.360   0.008  0.067
 high           2             17.361  -0.238  1.475   1.029  0.094


In [12]:
filt, ll = forward_filter(model, X_te, return_loglik=True)
smooth = model.predict_proba(X_te)   # CHỈ để so sánh, không dùng làm đặc trưng
filt_m, smooth_m = filt[:, order], smooth[:, order]
out(f"\n[B2] log-lik forward tự viết = {ll:.6f}; hmmlearn score = {model.score(X_te):.6f}; "
    f"chênh {abs(ll - model.score(X_te)):.2e}")
out(f"     |filtered - smoothed| tại điểm cuối: {np.abs(filt[-1] - smooth[-1]).max():.2e}")
d = np.abs(filt - smooth)[:-1]
out(f"     |filtered - smoothed| các điểm còn lại: trung bình {d.mean():.4f}, max {d.max():.4f}, "
    f"tỷ lệ bước chênh > 0,1: {(d.max(1) > 0.1).mean():.3f}")
acc_f = (filt_m.argmax(1) == z_te).mean()
acc_s = (smooth_m.argmax(1) == z_te).mean()
out(f"     Độ chính xác regime (argmax): filtered {acc_f:.3f} | smoothed {acc_s:.3f}"
    "\n     -> smoothed 'tốt hơn' chính là nhờ nhìn tương lai; không được dùng.")
p_hi = filt_m[:, -1]
out(f"     Đại diện P(high | x_1..t): trung bình khi z=high {p_hi[z_te == 2].mean():.3f}, "
    f"z=low {p_hi[z_te == 0].mean():.3f}")
true_ch = np.flatnonzero(np.diff(z_te == 2) != 0) + 1
lag = []
for c in true_ch[(z_te[true_ch] == 2)]:
    hit = np.flatnonzero(p_hi[c:c + 30] > 0.5)
    lag.append(hit[0] if hit.size else np.nan)
out(f"     Trễ bật P(high) > 0,5 khi vào regime high: trung vị {np.nanmedian(lag):.0f} bước, "
    f"max {np.nanmax(lag):.0f}, {len(lag)} lần vào")


[B2] log-lik forward tự viết = -1131.120019; hmmlearn score = -1131.120019; chênh 2.05e-12
     |filtered - smoothed| tại điểm cuối: 8.03e-14
     |filtered - smoothed| các điểm còn lại: trung bình 0.0132, max 0.7254, tỷ lệ bước chênh > 0,1: 0.045
     Độ chính xác regime (argmax): filtered 0.984 | smoothed 0.995
     -> smoothed 'tốt hơn' chính là nhờ nhìn tương lai; không được dùng.
     Đại diện P(high | x_1..t): trung bình khi z=high 0.972, z=low 0.000
     Trễ bật P(high) > 0,5 khi vào regime high: trung vị 0 bước, max 1, 13 lần vào


In [13]:
w = slice(0, 400)
tt = np.arange(len(X_te))[w]
fig, ax = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
ax[0].scatter(tt, X_te[w, 1], c=z_te[w], cmap="coolwarm", s=6)
ax[0].set_ylabel("log vol (giả lập)"); ax[0].set_title("HMM 3 state — màu: regime thật (xanh low, đỏ high)")
ax[1].fill_between(tt, 0, (z_te[w] == 2), color="tab:red", alpha=0.15, step="mid", label="thật: high")
ax[1].plot(tt, filt_m[w, -1], color="tab:red", lw=1, label="filtered P(high | x_1..t)")
ax[1].plot(tt, smooth_m[w, -1], color="k", lw=1, ls=":", label="smoothed P(high | x_1..T) — KHÔNG dùng")
ax[1].set_ylabel("P(high)"); ax[1].legend(loc="upper right", fontsize=8)
ax[2].stackplot(tt, filt_m[w].T, colors=["tab:blue", "tab:gray", "tab:red"], labels=["low", "mid", "high"])
ax[2].set_ylabel("filtered P(z_t)"); ax[2].set_xlabel("t (bước, tập test)")
ax[2].legend(loc="upper right", fontsize=8, ncol=3)
fig.tight_layout(); fig.savefig(FIG / "g2_hmm_sim.png", dpi=150); plt.close(fig)

In [14]:
section("C. Kiểm tra không nhìn tương lai (đổi dữ liệu sau t)")
t_cut = 600
x_alt = x0.copy(); x_alt[t_cut + 1:] = np.random.default_rng(1).normal(5, 3, len(x0) - t_cut - 1)
fa, fb = run_bocpd(x0, prior), run_bocpd(x_alt, prior)
db = np.abs(fa.to_numpy(float) - fb.to_numpy(float))
out(f"BOCPD, t = {t_cut}: max |chênh| mọi cột đến t = {db[:t_cut + 1].max():.1e}; "
    f"cp_recent sau t = {np.abs(fa.cp_recent - fb.cp_recent)[t_cut + 1:].max():.3f} (đối chứng: có đổi)")


## C. Kiểm tra không nhìn tương lai (đổi dữ liệu sau t)
BOCPD, t = 600: max |chênh| mọi cột đến t = 0.0e+00; cp_recent sau t = 0.991 (đối chứng: có đổi)


In [15]:
p_high_f = filt_m[:, -1]
t_h = int(np.flatnonzero((np.arange(len(X_te)) >= 600) & (p_high_f > 0.2) & (p_high_f < 0.8))[0])
X_alt = X_te.copy()
X_alt[t_h + 1:] = HMM_MEANS[0] + HMM_SDS[0] * np.random.default_rng(5).standard_normal(
    (len(X_te) - t_h - 1, X_te.shape[1]))
f_alt, s_alt = forward_filter(model, X_alt), model.predict_proba(X_alt)
out(f"HMM filtered, t = {t_h}: max |chênh| đến t = {np.abs(filt - f_alt)[:t_h + 1].max():.1e}")
out(f"HMM smoothed (đối chứng): max |chênh| đến t = {np.abs(smooth - s_alt)[:t_h + 1].max():.3f}"
    " -> smoothed rò rỉ tương lai")
out(f"  Tại t = {t_h}: filtered P(high) = {filt_m[t_h, -1]:.3f} (cả 2 trường hợp); smoothed P(high) "
    f"= {smooth_m[t_h, -1]:.3f} (gốc) vs {s_alt[:, order][t_h, -1]:.3f} (tương lai đổi sang low)")

HMM filtered, t = 646: max |chênh| đến t = 0.0e+00
HMM smoothed (đối chứng): max |chênh| đến t = 0.460 -> smoothed rò rỉ tương lai
  Tại t = 646: filtered P(high) = 0.573 (cả 2 trường hợp); smoothed P(high) = 0.982 (gốc) vs 0.522 (tương lai đổi sang low)


In [16]:
fig, ax = plt.subplots(2, 1, figsize=(11, 5.5))
r = slice(t_cut - 100, t_cut + 100)
ax[0].plot(np.arange(len(x0))[r], fa.cp_recent[r], label="dữ liệu gốc")
ax[0].plot(np.arange(len(x0))[r], fb.cp_recent[r], ls="--", label="dữ liệu sau t bị thay")
ax[0].axvline(t_cut, color="k", lw=0.8); ax[0].set_title(f"BOCPD cp_recent: trùng khít đến t = {t_cut}")
ax[0].legend(fontsize=8)
r = slice(t_h - 100, t_h + 40)
ii = np.arange(len(X_te))[r]
ax[1].plot(ii, filt_m[r, -1], color="tab:red", label="filtered — gốc")
ax[1].plot(ii, f_alt[:, order][r, -1], color="tab:red", ls="--", label="filtered — thay sau t")
ax[1].plot(ii, smooth_m[r, -1], color="k", lw=0.8, label="smoothed — gốc")
ax[1].plot(ii, s_alt[:, order][r, -1], color="k", lw=0.8, ls="--", label="smoothed — thay sau t")
ax[1].axvline(t_h, color="k", lw=0.8)
ax[1].scatter([t_h, t_h], [smooth_m[t_h, -1], s_alt[:, order][t_h, -1]], color="k", zorder=5, s=18)
ax[1].annotate(f"smoothed tại t: {smooth_m[t_h, -1]:.2f} → {s_alt[:, order][t_h, -1]:.2f}",
               (t_h, 0.5), xytext=(8, -30), textcoords="offset points", fontsize=8)
ax[1].set_title(f"HMM P(high): filtered trùng đến t = {t_h}, smoothed thì không")
ax[1].legend(fontsize=7, ncol=2)
fig.tight_layout(); fig.savefig(FIG / "g2_no_lookahead.png", dpi=150); plt.close(fig)

In [17]:
section("D. FILE ĐẦU RA")
for p in sorted(FIG.glob("g2_*.png")):
    out(" ", p.relative_to(ROOT))
(ROOT / "reports" / "g2_report.txt").write_text(buf.getvalue(), encoding="utf-8")
print("  reports/g2_report.txt")


## D. FILE ĐẦU RA
  reports\figures\g2_bocpd_sim.png
  reports\figures\g2_hmm_sim.png
  reports\figures\g2_no_lookahead.png
  reports/g2_report.txt
